# Source Data Analysis

This notebook profiles the raw assignment datasets before ingestion and modeling. The goal is to understand file formats, schemas, nulls, duplicates, date formats, relationship integrity, and cleanup requirements.

In [1]:
from pathlib import Path
from collections import Counter
from dateutil import parser
import json
import re

import pandas as pd

DATA_DIR = Path('data') if Path('data').exists() else Path('../data')
DATA_DIR.resolve()

PosixPath('/Users/rajat/Documents/MyProjects/iCIMS-assignment/icims-de-assignment/data')

## Load Source Files

In [3]:
jobs = pd.read_csv(DATA_DIR / 'jobs.csv')
applications = pd.read_csv(DATA_DIR / 'applications.csv')
education = pd.read_csv(DATA_DIR / 'education.csv')

with open(DATA_DIR / 'candidates.json') as f:
    candidates = pd.DataFrame(json.load(f))

workflow_events = pd.read_json(DATA_DIR / 'workflow_events.jsonl', lines=True)

sources = {
    'jobs.csv': jobs,
    'applications.csv': applications,
    'education.csv': education,
    'candidates.json': candidates,
    'workflow_events.jsonl': workflow_events,
}

pd.DataFrame([
    {
        'file': name,
        'rows': len(df),
        'columns': len(df.columns),
        'column_names': ', '.join(df.columns),
    }
    for name, df in sources.items()
])

,file,rows,columns,column_names
0,jobs.csv,500,5,"job_id, title, department, posted_date, status"
1,applications.csv,5000,4,"application_id, job_id, candidate_id, apply_date"
2,education.csv,2000,4,"candidate_id, degree, institution, year"
3,candidates.json,2001,6,"candidate_id, first_name, last_name, email, ph..."
4,workflow_events.jsonl,16769,4,"application_id, old_status, new_status, event_..."


## Schema, Nulls, And Duplicate Checks

In [4]:
def profile_dataframe(name, df):
    return pd.DataFrame({
        'file': name,
        'column': df.columns,
        'dtype': [str(df[col].dtype) for col in df.columns],
        'null_count': [int(df[col].isna().sum()) for col in df.columns],
        'distinct_count': [int(df[col].astype(str).nunique(dropna=False)) for col in df.columns],
    })

schema_profile = pd.concat(
    [profile_dataframe(name, df) for name, df in sources.items()],
    ignore_index=True,
)
schema_profile

,file,column,dtype,null_count,distinct_count
0,jobs.csv,job_id,object,0,500
1,jobs.csv,title,object,0,7
2,jobs.csv,department,object,48,7
3,jobs.csv,posted_date,object,0,329
4,jobs.csv,status,object,0,3
5,applications.csv,application_id,object,0,5000
6,applications.csv,job_id,object,0,500
7,applications.csv,candidate_id,object,0,1841
8,applications.csv,apply_date,object,0,843
9,education.csv,candidate_id,object,0,2000


In [5]:
duplicate_summary = []
for name, df in sources.items():
    duplicate_summary.append({
        'file': name,
        'full_duplicate_rows': int(df.astype(str).duplicated().sum()),
    })

pd.DataFrame(duplicate_summary)

,file,full_duplicate_rows
0,jobs.csv,0
1,applications.csv,0
2,education.csv,0
3,candidates.json,0
4,workflow_events.jsonl,0


## Date Format Profiling

In [6]:
def date_pattern(value):
    value = str(value)
    if re.match(r'^\d{4}-\d{2}-\d{2}$', value):
        return 'YYYY-MM-DD'
    if re.match(r'^\d{4}/\d{2}/\d{2}$', value):
        return 'YYYY/MM/DD'
    if re.match(r'^\d{4}\.\d{2}\.\d{2}$', value):
        return 'YYYY.MM.DD'
    if re.match(r'^\d{2}-[A-Za-z]{3}-\d{4}$', value):
        return 'DD-Mon-YYYY'
    if re.match(r'^[A-Za-z]+ \d{1,2}, \d{4}$', value):
        return 'Month D, YYYY'
    if re.match(r'^\d{4}-\d{2}-\d{2}T', value):
        return 'ISO timestamp'
    return 'other'

date_columns = {
    'jobs.csv': ('posted_date', jobs['posted_date']),
    'applications.csv': ('apply_date', applications['apply_date']),
    'workflow_events.jsonl': ('event_timestamp', workflow_events['event_timestamp']),
}

date_profile_rows = []
for file_name, (column_name, series) in date_columns.items():
    counts = Counter(series.dropna().map(date_pattern))
    for pattern, count in counts.items():
        date_profile_rows.append({
            'file': file_name,
            'column': column_name,
            'format': pattern,
            'count': count,
        })

pd.DataFrame(date_profile_rows).sort_values(['file', 'count'], ascending=[True, False])

,file,column,format,count
5,applications.csv,apply_date,YYYY-MM-DD,4226
6,applications.csv,apply_date,DD-Mon-YYYY,216
8,applications.csv,apply_date,"Month D, YYYY",197
9,applications.csv,apply_date,YYYY.MM.DD,183
7,applications.csv,apply_date,YYYY/MM/DD,178
2,jobs.csv,posted_date,YYYY-MM-DD,423
1,jobs.csv,posted_date,YYYY.MM.DD,25
0,jobs.csv,posted_date,YYYY/MM/DD,18
4,jobs.csv,posted_date,DD-Mon-YYYY,18
3,jobs.csv,posted_date,"Month D, YYYY",16


## Categorical Distributions

In [7]:
display(jobs['department'].value_counts(dropna=False).rename_axis('department').reset_index(name='count'))
display(jobs['status'].value_counts(dropna=False).rename_axis('status').reset_index(name='count'))
display(education['degree'].value_counts(dropna=False).rename_axis('degree').reset_index(name='count'))
display(workflow_events['new_status'].value_counts(dropna=False).rename_axis('new_status').reset_index(name='count'))

,department,count
0,Marketing,90
1,Product,85
2,Engineering,76
3,Sales,74
4,Finance,65
5,HR,62
6,NaN,48


,status,count
0,Open,178
1,Closed,169
2,Draft,153


,degree,count
0,BS,683
1,MS,675
2,PhD,642


,new_status,count
0,Applied,5000
1,Screening,3811
2,Interview,2646
3,Offer,1566
4,Withdrawn,1263
5,Rejected,1250
6,Hired,1233


## Candidate Skill Analysis

In [8]:
skill_counts = candidates['skills'].apply(lambda x: len(x) if isinstance(x, list) else 0)
all_skills = Counter(
    skill
    for skills in candidates['skills']
    for skill in (skills if isinstance(skills, list) else [])
)

print('Skill count min:', skill_counts.min())
print('Skill count max:', skill_counts.max())
print('Skill count average:', round(skill_counts.mean(), 2))

pd.DataFrame(all_skills.most_common(10), columns=['skill', 'count'])

Skill count min: 1
Skill count max: 5
Skill count average: 3.01


,skill,count
0,Communication,652
1,AWS,619
2,Node.js,612
3,React,603
4,Excel,602
5,Docker,601
6,Java,600
7,Kubernetes,585
8,SQL,583
9,Python,568


## Relationship Integrity Checks

In [9]:
relationship_checks = [
    {
        'check': 'Application job_id values missing from jobs',
        'count': len(set(applications['job_id']) - set(jobs['job_id'])),
    },
    {
        'check': 'Application candidate_id values missing from candidates',
        'count': len(set(applications['candidate_id']) - set(candidates['candidate_id'])),
    },
    {
        'check': 'Education candidate_id values missing from candidates',
        'count': len(set(education['candidate_id']) - set(candidates['candidate_id'])),
    },
    {
        'check': 'Workflow application_id values missing from applications',
        'count': len(set(workflow_events['application_id']) - set(applications['application_id'])),
    },
    {
        'check': 'Candidates with no applications',
        'count': len(set(candidates['candidate_id']) - set(applications['candidate_id'])),
    },
    {
        'check': 'Candidates with no education',
        'count': len(set(candidates['candidate_id']) - set(education['candidate_id'])),
    },
    {
        'check': 'Candidates who applied to more than 3 jobs',
        'count': int((applications.groupby('candidate_id')['job_id'].nunique() > 3).sum()),
    },
]

pd.DataFrame(relationship_checks)

,check,count
0,Application job_id values missing from jobs,0
1,Application candidate_id values missing from c...,0
2,Education candidate_id values missing from can...,0
3,Workflow application_id values missing from ap...,0
4,Candidates with no applications,160
5,Candidates with no education,1
6,Candidates who applied to more than 3 jobs,506


## Hired-Before-Applied Anomaly

In [10]:
def parse_date(value):
    try:
        return pd.Timestamp(parser.parse(str(value))).normalize()
    except Exception:
        return pd.NaT

applications_parsed = applications.assign(
    apply_date_parsed=applications['apply_date'].map(parse_date)
)[['application_id', 'apply_date', 'apply_date_parsed']]

events_parsed = workflow_events.assign(
    event_timestamp_parsed=workflow_events['event_timestamp'].map(parse_date)
)

events_with_apply_date = events_parsed.merge(applications_parsed, on='application_id', how='left')

hired_before_applied = events_with_apply_date[
    (events_with_apply_date['new_status'] == 'Hired')
    & (events_with_apply_date['event_timestamp_parsed'] < events_with_apply_date['apply_date_parsed'])
]

hired_before_applied[[
    'application_id',
    'old_status',
    'new_status',
    'event_timestamp',
    'apply_date',
]]

,application_id,old_status,new_status,event_timestamp,apply_date
16768,2391ab47-f15a-4799-a890-64e2deac7190,Interview,Hired,2025-11-08T00:00:00,2025-11-13


## Cleanup Rules Derived From Profiling

- Preserve raw data and add ingestion metadata.
- Parse mixed date formats in staging.
- Normalize status, department, degree, and email fields.
- Deduplicate using business keys.
- Generate a surrogate key for workflow events.
- Validate relationship integrity across applications, jobs, candidates, education, and workflow events.
- Flag hired-before-applied records and exclude them from time-to-hire calculations.